In [1]:
# Cell 1, Impor Pustaka & Helper

# Impor Pustaka Eksternal
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options  
from webdriver_manager.chrome import ChromeDriverManager

# Impor Modul Lokal
# --- PENTING!!!: Impor fungsi dari file 'scraper_helper.py' ---
from scraper_helper import scrape_year_data

(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)


In [ ]:
# Cell 2, Inisiasi Driver (Jalankan Satu Kali)

def setup_stealth_driver():
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-web-security")
    options.add_argument("--disable-features=VizDisplayCompositor")

    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    # Hapus window.navigator.webdriver
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    return driver


print("Memulai Chrome Driver dengan stealth mode...")

driver = setup_stealth_driver()
driver.maximize_window()

driver.get("https://x.com/login")
print("Driver dimulai. Browser telah terbuka. Silakan login secara manual.")

Memulai Chrome Driver dengan stealth mode...


In [2]:
# Cell 3: Pusat Kendali Scraping (Eksekusi per Tahun)

TAHUN_TARGET = 2015
MAX_SCROLLS = 1000

if 'driver' in locals() and driver.session_id:
    print(f"======= MEMULAI PROSES SCRAPING TAHUN {TAHUN_TARGET} =======")

    try:
        # --- HANYA MEMANGGIL FUNGSI ---
        # Semua logika rumit ada di scraper_helper.py
        scrape_year_data(TAHUN_TARGET, driver, max_scrolls=MAX_SCROLLS)

        print(f"\n======= SELESAI SCRAPING TAHUN {TAHUN_TARGET} =======\n")
    except Exception as e:
        print(f"\n--- Terjadi Error saat scraping tahun {TAHUN_TARGET}: {e} ---")
        print("Proses dihentikan. Driver tetap terbuka untuk inspeksi.")

else:
    print("ERROR: Driver tidak ditemukan. Harap jalankan 'Cell 2' terlebih dahulu dan selesaikan login manual.")

ERROR: Driver tidak ditemukan. Harap jalankan 'Cell 2' terlebih dahulu dan selesaikan login manual.


In [4]:
# Cell 4: Tutup Driver 
try:
    driver.quit()
    print("Driver telah berhasil ditutup.")
except Exception as e:
    print(f"Gagal menutup driver (mungkin sudah ditutup): {e}")

Driver telah berhasil ditutup.


In [26]:
# Cell: Pengecekan Total Data
import csv
import os

tahun_target = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]
total_data_global = 0

print("--- Hasil Total Data Scraping (per File) ---")

for year in tahun_target:
    filename = f"aspirasi_dpr_{year}.csv"

    # Cek apakah file-nya ada
    if os.path.exists(filename):
        try:
            with open(filename, 'r', newline='', encoding='utf-8') as f:
                # Membaca semua baris
                reader = csv.reader(f)
                # Menghitung jumlah baris (dikurangi 1 untuk header)
                data_count = len(list(reader)) - 1

                # Pastikan data_count tidak negatif jika file kosong
                if data_count < 0:
                    data_count = 0

                print(f"Dataset Tahun {year}: {data_count} baris")
                total_data_global += data_count
        except Exception as e:
            print(f"Gagal membaca file {filename}: {e}")
    else:
        # Beri tahu jika file-nya tidak ditemukan
        print(f"File {filename} tidak ditemukan (dilewati).")

print("\n")
print(f"TOTAL DATA KESELURUHAN: {total_data_global} baris")

--- Hasil Total Data Scraping (per File) ---
Dataset Tahun 2016: 6204 baris
Dataset Tahun 2017: 5290 baris
Dataset Tahun 2018: 2387 baris
Dataset Tahun 2019: 392 baris
Dataset Tahun 2020: 517 baris
Dataset Tahun 2021: 636 baris
Dataset Tahun 2022: 30 baris
Dataset Tahun 2023: 74 baris
Dataset Tahun 2024: 633 baris
Dataset Tahun 2025: 216 baris


TOTAL DATA KESELURUHAN: 16379 baris


In [3]:
import pandas as pd
import glob
import os

In [4]:
files = glob.glob('aspirasi_dpr_*.csv')

In [5]:
data_list = []

for f in files:
    # Membaca file
    df = pd.read_csv(f)
    
    # Mengambil tahun dari nama file (misal: dari 'aspirasi_dpr_2016.csv' ambil '2016')
    tahun = f.split('_')[-1].split('.')[0]
    df['tahun'] = tahun
    
    data_list.append(df)

In [6]:
master_df = pd.concat(data_list, ignore_index=True)

In [7]:
master_df.to_csv('master_dataset_aspirasi_dpr.csv', index=False)

print(f"Total data setelah digabung: {len(master_df)} baris.")

Total data setelah digabung: 16379 baris.
